14 — Terrain EDA: compare features, then inspect sites

Start feature by feature across sites; use maps to explain differences and extremes.
This notebook supports terrain analog-search design. It does not rank mission
suitability or fit a model. Environment EDA is a separate next step because its
units, spatial support and time bins differ from terrain.

Questions:
1. How do terrain distributions vary across the three Earth and seven lunar sites?
2. How much do Earth screening and lunar image-support policies change the population?
3. Which channels are redundant, and where are extreme values located?

Statistics use **all cells with all five features present** within each stated
population. Feature-specific missingness uses the ROI as its denominator.
Correlations alone use a reproducible, capped sample of those joint-valid cells.
Equal 30 m spacing does not imply equal effective resolution. Elevation datums
differ across bodies; absolute heights are context, not an analog distance.
Spatially neighboring cells are correlated: there are no independent-sample
p-values or confidence intervals here. No values are imputed or deleted.

Run all cells with the project kernel. Each run writes a new diagnostic folder.
Raw and processed inputs are read-only. A run is complete only when its manifest
says complete. Pin different completed run IDs explicitly before rerunning.

In [ ]:
import hashlib
import json
import sys
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path
from typing import Any

import matplotlib
import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import rasterio
from IPython.display import Image, Markdown, display
from matplotlib.figure import Figure

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'outline.md').exists())
sys.path.insert(0, str(ROOT / 'notebooks'))
from inspection import check_input, file_hash, verify_run

NOTEBOOK = ROOT / 'notebooks/exploration/14_terrain_feature_eda.ipynb'
RUN_IDS = {
    'earth_terrain_landcover': '20261028T063142045472Z_9d6d3b',
    'lunar_terrain_views': '20261028T084443441751Z_1a794f',
}
FEATURES = {
    'elevation_m': 'Elevation (m; body-specific datum)',
    'relative_elevation_m': 'Region/view-relative elevation (m)',
    'slope_deg': 'Slope (degrees)',
    'laplacian_per_m': 'Laplacian (1/m)',
    'local_elevation_std_m': 'Local elevation SD (m; 330 m window)',
}
FEATURE_NAMES = list(FEATURES)
SEED = 20261028
CORRELATION_SAMPLE_CAP = 50000
QUANTILES = [0, .01, .05, .25, .5, .75, .95, .99, 1]
QUANTILE_NAMES = ['min', 'p01', 'p05', 'p25', 'median', 'p75', 'p95', 'p99', 'max']
OUT = ROOT / 'notebooks/diagnostics' / (
    'terrain-eda-' + datetime.now(UTC).strftime('%Y%m%dT%H%M%S%fZ'))
OUT.mkdir(parents=True, exist_ok=False)
expected: dict[str, str] = {}
evidence: dict[str, str] = {}
runs: dict[str, Path] = {}
for stage, run_id in RUN_IDS.items():
    run = ROOT / 'data/processed' / stage / run_id
    expected.update(verify_run(run))
    evidence[str(run / 'provenance.json')] = file_hash(run / 'provenance.json')
    runs[stage] = run
book = nbformat.read(NOTEBOOK, as_version=4)
source_text = '\n'.join(cell.source for cell in book.cells)
provenance: dict[str, Any] = {
    'status': 'running', 'notebook': str(NOTEBOOK.relative_to(ROOT)),
    'notebook_source_sha256': hashlib.sha256(source_text.encode()).hexdigest(),
    'helpers': {'inspection.py': file_hash(ROOT / 'notebooks/inspection.py')},
    'runs': RUN_IDS, 'seed': SEED, 'correlation_sample_cap': CORRELATION_SAMPLE_CAP,
    'quantiles': QUANTILES, 'extreme_flag': 'outside Q1 - 3 IQR or Q3 + 3 IQR',
    'python': sys.version,
    'software': {name: version(name) for name in
                 ['numpy', 'pandas', 'scipy', 'rasterio', 'matplotlib', 'nbformat']},
}
(OUT / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .2})


def read_layer(path: Path) -> tuple[np.ndarray, tuple[Any, ...]]:
    """Verify provenance and return decoded values plus their grid identity."""
    with rasterio.open(check_input(path, expected, evidence)) as src:
        assert src.count == 1 and src.scales == (1.0,) and src.offsets == (0.0,), path
        assert src.crs is not None and src.transform.a == 30 and src.transform.e == -30
        assert src.transform.b == src.transform.d == 0
        values = src.read(1, masked=True).astype('float32').filled(np.nan)
        assert not np.isinf(values).any(), path
        return values, (src.shape, src.transform, src.crs)


def save_figure(fig: Figure, name: str) -> None:
    fig.savefig(OUT / f'{name}.png', bbox_inches='tight')
    fig.savefig(OUT / f'{name}.pdf', bbox_inches='tight')
    display(Image(filename=str(OUT / f'{name}.png')))
    plt.close(fig)


print('Pinned runs:', RUN_IDS)
print('Diagnostic folder:', OUT)

# Populations and complete-feature coverage

Earth: compare all joint-valid ROI cells with those also passing the existing
natural-terrain screen. The screen is a land-cover policy, not proof of access.
The all-ROI population includes water/zero-height ocean cells around Lanzarote;
it is a screening diagnostic, not a land-only terrain sample.

Moon: compare the full published terrain with the strict image-supported view.
The stored complete-feature masks must agree exactly with finite-feature checks.
Full-context heights in poorly illuminated areas can be prior-dominated.
The strict mask also excludes neighborhoods around unsupported pixels.
All areas below are **projected grid areas**, calculated as cell count × 900 m².

In [ ]:
earth_folders = sorted((runs['earth_terrain_landcover'] / 'earth').glob('*/grid_30m'))
moon_folders = sorted((runs['lunar_terrain_views'] / 'moon').iterdir())
assert len(earth_folders) == 3 and len(moon_folders) == 7
coverage_rows: list[dict[str, Any]] = []
summary_rows: list[dict[str, Any]] = []
correlation_rows: list[dict[str, Any]] = []
extreme_rows: list[dict[str, Any]] = []
sensitivity_rows: list[dict[str, Any]] = []
site_specs = [('earth', p.parent.name, p) for p in earth_folders]
site_specs += [('moon', p.name, p) for p in moon_folders]
map_paths: list[Path] = []

for site_index, (body, region, folder) in enumerate(site_specs):
    populations: dict[str, tuple[dict[str, np.ndarray], np.ndarray]] = {}
    if body == 'earth':
        screen, grid = read_layer(folder / 'natural_screen_mask.tif')
        inside = np.isfinite(screen)
        assert set(np.unique(screen[inside])) <= {0, 1}
        layers = {}
        for feature in FEATURE_NAMES:
            layers[feature], feature_grid = read_layer(folder / f'{feature}.tif')
            assert feature_grid == grid
        joint = inside & np.logical_and.reduce([np.isfinite(a) for a in layers.values()])
        populations['all_complete'] = (layers, joint)
        populations['natural_screen'] = (layers, joint & (screen == 1))
    else:
        footprint, grid = read_layer(folder / 'quality/source_coverage_fraction.tif')
        inside = np.isfinite(footprint) & (footprint > 0)
        for view in ['full_context', 'image_supported']:
            layers = {}
            for feature in FEATURE_NAMES:
                layers[feature], feature_grid = read_layer(folder / view / f'{feature}.tif')
                assert feature_grid == grid
            joint = inside & np.logical_and.reduce([np.isfinite(a) for a in layers.values()])
            saved_mask, mask_grid = read_layer(folder / 'quality' / f'{view}_complete_feature_mask.tif')
            assert mask_grid == grid and np.array_equal(joint, saved_mask == 1), (region, view)
            populations[view] = (layers, joint)

    shape, transform, crs = grid
    for pop_index, (population, (layers, eligible)) in enumerate(populations.items()):
        n = int(eligible.sum())
        assert n > 1, (region, population)
        coverage_rows.append({
            'body': body, 'region': region, 'population': population,
            'roi_cells': int(inside.sum()), 'joint_cells': n,
            'joint_pct_of_roi': 100 * n / int(inside.sum()),
            'projected_area_km2': n * .0009,
        })
        valid_indices = np.flatnonzero(eligible)
        rng = np.random.default_rng(SEED + site_index * 10 + pop_index)
        chosen = rng.choice(valid_indices, size=min(n, CORRELATION_SAMPLE_CAP), replace=False)
        sample = pd.DataFrame({f: layers[f].ravel()[chosen] for f in FEATURE_NAMES})
        correlation = sample.corr(method='spearman')
        for i, first in enumerate(FEATURE_NAMES):
            for second in FEATURE_NAMES[i + 1:]:
                correlation_rows.append({
                    'body': body, 'region': region, 'population': population,
                    'first': first, 'second': second,
                    'spearman_rho': float(correlation.loc[first, second]),
                    'sample_cells': len(chosen), 'population_cells': n,
                    'seed': SEED + site_index * 10 + pop_index,
                })
        for feature, values in layers.items():
            selected = values[eligible]
            quantiles = np.quantile(selected, QUANTILES)
            stats = dict(zip(QUANTILE_NAMES, map(float, quantiles), strict=True))
            iqr = stats['p75'] - stats['p25']
            low, high = stats['p25'] - 3 * iqr, stats['p75'] + 3 * iqr
            flags = (selected < low) | (selected > high)
            summary_rows.append({
                'body': body, 'region': region, 'population': population,
                'feature': feature, 'joint_cells': n,
                'feature_missing_roi_cells': int((inside & ~np.isfinite(values)).sum()),
                'roi_cells': int(inside.sum()), **stats,
                'mean': float(np.mean(selected, dtype='float64')),
                'sd': float(np.std(selected, dtype='float64')),
                'outer_fence_low': low, 'outer_fence_high': high,
                'flagged_cells': int(flags.sum()), 'flagged_pct': 100 * float(flags.mean()),
            })
            for direction, position in [('min', int(np.argmin(selected))), ('max', int(np.argmax(selected)))]:
                flat_index = int(valid_indices[position])
                row, col = np.unravel_index(flat_index, shape)
                x, y = transform * (int(col) + .5, int(row) + .5)
                extreme_rows.append({
                    'body': body, 'region': region, 'population': population,
                    'feature': feature, 'direction': direction, 'value': float(selected[position]),
                    'outer_fence_flag': bool(flags[position]), 'row': int(row), 'col': int(col),
                    'x_m': x, 'y_m': y, 'crs_wkt': crs.to_wkt(),
                })
        assert (layers['slope_deg'][eligible] >= 0).all()
        assert (layers['slope_deg'][eligible] <= 90).all()
        assert (layers['local_elevation_std_m'][eligible] >= 0).all()

    if body == 'moon':
        full, full_mask = populations['full_context']
        strict, strict_mask = populations['image_supported']
        common = full_mask & strict_mask
        assert common.any()
        for feature in FEATURE_NAMES:
            paired_delta = full[feature][common].astype('float64') - strict[feature][common]
            full_all_median = float(np.median(full[feature][full_mask]))
            full_common_median = float(np.median(full[feature][common]))
            sensitivity_rows.append({
                'region': region, 'feature': feature,
                'full_cells': int(full_mask.sum()), 'strict_cells': int(strict_mask.sum()),
                'common_cells': int(common.sum()),
                'strict_cells_outside_full_complete': int((strict_mask & ~full_mask).sum()),
                'full_all_median': full_all_median, 'full_common_median': full_common_median,
                'full_all_minus_common_median': full_all_median - full_common_median,
                'strict_common_median': float(np.median(strict[feature][common])),
                'paired_median_delta': float(np.median(paired_delta)),
                'paired_p99_abs_delta': float(np.quantile(np.abs(paired_delta), .99)),
                'paired_max_abs_delta': float(np.abs(paired_delta).max()),
            })

    # Site maps show the primary population; extreme coordinates retain the full grid.
    primary = 'natural_screen' if body == 'earth' else 'full_context'
    layers, eligible = populations[primary]
    stride = max(1, int(np.ceil(max(shape) / 700)))
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout='constrained')
    for ax, feature in zip(axes, ['slope_deg', 'local_elevation_std_m'], strict=True):
        image_values = np.where(eligible, layers[feature], np.nan)[::stride, ::stride]
        vmax = 45 if feature == 'slope_deg' else float(np.quantile(layers[feature][eligible], .99))
        cmap = matplotlib.colormaps['viridis'].copy()
        cmap.set_bad('#dddddd')
        plot = ax.imshow(image_values, vmin=0, vmax=max(vmax, 1e-12), cmap=cmap,
                         extent=(-stride / 2, (image_values.shape[1] - .5) * stride,
                                 (image_values.shape[0] - .5) * stride, -stride / 2),
                         interpolation='nearest')
        ax.set_xlim(-.5, shape[1] - .5)
        ax.set_ylim(shape[0] - .5, -.5)
        fig.colorbar(plot, ax=ax, label=FEATURES[feature], shrink=.7, extend='max')
        for record in extreme_rows:
            if (record['region'] == region and record['population'] == primary
                    and record['feature'] == feature and record['direction'] == 'max'):
                ax.scatter(record['col'], record['row'], marker='x', s=65, c='red')
        ax.set(title=FEATURES[feature], xlabel='Column (30 m grid)', ylabel='Row (30 m grid)')
    fig.suptitle(f'{region} | {primary}\nGray: excluded/missing; red ×: exact maximum; preview stride {stride}')
    name = f'map_{region}'
    fig.savefig(OUT / f'{name}.png', bbox_inches='tight')
    fig.savefig(OUT / f'{name}.pdf', bbox_inches='tight')
    plt.close(fig)
    map_paths.append(OUT / f'{name}.png')
    print(region, {p: int(m.sum()) for p, (_, m) in populations.items()})

coverage = pd.DataFrame(coverage_rows)
summaries = pd.DataFrame(summary_rows)
correlations = pd.DataFrame(correlation_rows)
extremes = pd.DataFrame(extreme_rows)
sensitivity = pd.DataFrame(sensitivity_rows)
for name, frame in [('coverage', coverage), ('feature_statistics', summaries),
                    ('correlations', correlations), ('extreme_locations', extremes),
                    ('lunar_sensitivity', sensitivity)]:
    frame.to_csv(OUT / f'{name}.csv', index=False)
assert len(coverage) == 20 and len(summaries) == 100 and len(sensitivity) == 35
display(coverage)

# Feature-first comparison

Dots show medians; thick bars show the interquartile range; thin bars show P05–P95.
Each row has equal visual weight, regardless of site size. Exact min/max and
P01/P99 remain in feature_statistics.csv; no data are clipped for statistics.
Earth rows use natural-screened complete terrain; lunar rows use full-context
complete terrain. These populations have different selection policies.
Relative elevation is centered on a region/view-specific median before this
population selection, so its displayed median need not be zero.

In [ ]:
primary_populations = ['natural_screen', 'full_context']
primary_stats = pd.DataFrame(summaries.loc[summaries['population'].isin(primary_populations)]).copy()
for feature, label in FEATURES.items():
    rows = pd.DataFrame(primary_stats.loc[primary_stats['feature'] == feature]).sort_values(['body', 'region'])
    fig, ax = plt.subplots(figsize=(11, 5), layout='constrained')
    for y, row in enumerate(rows.to_dict('records')):
        color = '#007b83' if row['body'] == 'earth' else '#9d5b25'
        ax.plot([row['p05'], row['p95']], [y, y], color=color, linewidth=1.5)
        ax.plot([row['p25'], row['p75']], [y, y], color=color, linewidth=6)
        ax.plot(row['median'], y, 'o', color='black', markersize=4)
    ax.set_yticks(range(len(rows)), [f"{r['body']}: {r['region']}" for r in rows.to_dict('records')])
    ax.invert_yaxis()
    ax.set(xlabel=label, title='Exact per-site quantiles | dot: median; thick: IQR; thin: P05–P95')
    save_figure(fig, f'distribution_{feature}')
display(primary_stats[['region', 'feature', 'median', 'p95', 'min', 'max', 'flagged_pct']])

# Policy sensitivity and redundant channels

Earth pairs isolate land-cover screening with the same terrain values.
For the Moon, full_all_minus_common_median isolates population selection
while holding the full-context terrain fixed. Paired differences use the
**same coordinates**, so they measure changes between values in the two views.
Different relative-elevation zero points are expected.

Spearman correlations are descriptive, within-site comparisons on at most
50,000 cells per population. The sample seed and size are saved for every pair.
Undefined correlations (constant channels) stay missing. Do not pool all sites
and interpret a large site's sample size as stronger scientific evidence.

In [ ]:
earth_compare = summaries[summaries['body'] == 'earth'].pivot(
    index=['region', 'feature'], columns='population', values='median')
earth_compare['screened_minus_all_median'] = earth_compare['natural_screen'] - earth_compare['all_complete']
earth_compare.to_csv(OUT / 'earth_screen_sensitivity.csv')
display(earth_compare)
display(sensitivity[['region', 'feature', 'full_all_minus_common_median',
                     'paired_median_delta', 'paired_p99_abs_delta']])
rho = correlations[correlations['population'].isin(primary_populations)].copy()
rho['pair'] = rho['first'] + ' / ' + rho['second']
matrix = rho.pivot(index='region', columns='pair', values='spearman_rho')
fig, ax = plt.subplots(figsize=(12, 7), layout='constrained')
plot = ax.imshow(matrix.to_numpy(dtype=float), cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=70, ha='right', fontsize=8)
ax.set_yticks(range(len(matrix)), matrix.index, fontsize=8)
ax.set_title('Within-site Spearman correlations | primary populations | sampled cells')
fig.colorbar(plot, ax=ax, label='Spearman rho')
save_figure(fig, 'feature_correlations')

# Site maps and extreme-value review

Maps below show each site's primary population. Raster previews are subsampled;
they are not quantitative area measurements. Red crosses locate the exact
full-resolution maximum of slope or local elevation SD (first row-major cell
if tied). Coordinates, CRS, min/max values and the exploratory 3×IQR flag are
saved for all five features and all populations in extreme_locations.csv.
Slope uses a common 0–45 degree color scale. Local elevation SD uses 0 to that
site's P99; colors above the display maximum saturate, but statistics retain
the original values. Compare the numeric legends when viewing different sites.
A flagged tail may represent real terrain. Review its neighborhood and source
quality before treating it as a defect; no outliers are removed.

In [ ]:
for path in map_paths:
    display(Image(filename=str(path), width=1050))
display(extremes[extremes['outer_fence_flag']][
    ['region', 'population', 'feature', 'direction', 'value', 'row', 'col']])

# Conclusions and concrete next steps

These conclusions describe this pinned dataset. They are not accuracy estimates,
final feature-selection decisions or a ranking of Earth analog sites.

In [ ]:
# Read the Markdown notes below for the standalone report export.
report_notebook = json.loads((ROOT / 'notebooks/exploration/14_terrain_feature_eda.ipynb').read_text(encoding="utf-8"))
report_notes = {entry["metadata"]["report_section"]: "".join(entry["source"])
                for entry in report_notebook["cells"]
                if entry["cell_type"] == "markdown" and "report_section" in entry["metadata"]}
slope_stats = pd.DataFrame(primary_stats.loc[primary_stats['feature'] == 'slope_deg']).sort_values('median')
smoothest = slope_stats.iloc[0]
steepest = slope_stats.iloc[-1]
lunar_slope = pd.DataFrame(sensitivity.loc[sensitivity['feature'] == 'slope_deg'])
strongest_selection = lunar_slope.iloc[
    int(np.argmax(np.abs(np.asarray(lunar_slope['full_all_minus_common_median']))))]
roughness_pairs = rho[(rho['first'] == 'slope_deg') & (rho['second'] == 'local_elevation_std_m')]
pair_values = np.asarray(roughness_pairs['spearman_rho'], dtype=float)
finite_rho = pair_values[np.isfinite(pair_values)]
coverage_moon = coverage[coverage['body'] == 'moon'].pivot(
    index='region', columns='population', values='joint_cells')
strict_retention = 100 * coverage_moon['image_supported'] / coverage_moon['full_context']
report = [
    '# Terrain EDA: findings and actions',
    '',
    f'Compared {len(site_specs)} sites, {len(coverage)} populations and five terrain features. Quantiles use all joint-valid cells; correlations use seeded samples capped at {CORRELATION_SAMPLE_CAP:,} cells per population.',
    '',
    f"- Median slope spans {smoothest['median']:.2f}° ({smoothest['region']}) to {steepest['median']:.2f}° ({steepest['region']}) in the primary populations.",
    f'- Strict lunar complete-feature cell counts are {strict_retention.min():.1f}%–{strict_retention.max():.1f}% of full-context counts. These are count ratios; common-pixel counts are recorded separately.',
    f"- The largest absolute lunar slope median shift from population selection is {strongest_selection['full_all_minus_common_median']:+.2f}° ({strongest_selection['region']}; full population minus common population, both evaluated using full-context terrain).",
    f"- Maximum site-level P99 absolute paired slope difference is {lunar_slope['paired_p99_abs_delta'].max():.4g}° on common coordinates.",
    f'- Within-site slope/local-elevation-SD Spearman rho ranges from {finite_rho.min():.3f} to {finite_rho.max():.3f} in the primary populations. This measures association, not interchangeable physical meaning.',
    *report_notes['narrative-d7e9469d-1'].split('\n'),
]
report_text = '\n'.join(report) + '\n'
(OUT / 'REPORT.md').write_text(report_text, encoding='utf-8')
provenance['inputs'] = {str(Path(p).relative_to(ROOT)): digest for p, digest in evidence.items()}
provenance['outputs'] = {
    str(p.relative_to(OUT)): file_hash(p) for p in sorted(OUT.iterdir())
    if p.is_file() and p.name != 'provenance.json'}
provenance['status'] = 'complete'
provenance['completed_utc'] = datetime.now(UTC).isoformat()
(OUT / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
display(Markdown(report_text))
print('Complete:', OUT)


## Actions

1. Carry both lunar views and their quality masks forward. Compare population selection separately from numerical differences before choosing a training policy.
2. Use relative elevation for terrain-shape experiments; keep absolute elevation as context. It differs by a region/view constant and uses body-specific datums. Do not include both as independent evidence in a similarity score.
3. Inspect the saved extreme coordinates and source neighborhoods. The 3×IQR flags identify tails, not bad data; retain values unless a source defect is shown.
4. Next compare patch-level relief, slope and texture at the existing 300/600/960 m scales. Decide the spatial evaluation split before fitting normalization or selecting features. Site-level marginal distributions cannot establish analog matches.
5. Run environment EDA separately: documented illumination fractions first; then climate and Diviner using their native temporal/spatial support and coverage. Keep unresolved-unit layers out of quantitative combinations.

## Reading the outputs

- feature_statistics.csv: exact quantiles, mean/SD, missingness and exploratory tail flags.
- coverage.csv: explicit population sizes and projected grid area.
- lunar_sensitivity.csv: common-pixel differences versus population selection.
- earth_screen_sensitivity.csv: effect of the land-cover screen on medians.
- correlations.csv: within-site pair correlations, seeds and sample sizes.
- extreme_locations.csv: full-resolution min/max coordinates and their CRS.
- PNG/PDF plots: feature comparisons, correlations and ten site maps.

Uncertainty remains in effective resolution, prior-dominated lunar heights, source registration and the relevance of these descriptors to a specific mission. No missing values were filled; no scores, model training or claims of safety were produced.